# PBO — Perturbación de restricciones

Sesión independiente para **constraint-perturbation**. Permite elegir un trabajo de la matriz (MVC, MIS o CA) para este runtime. Ejecuta un único trabajo de esta técnica; usa un notebook y runtime independiente por cada técnica para entrenarlas simultáneamente. La matriz del repositorio excluye baseline, polarity y permutation ya entrenados; cada trabajo conserva su checkpoint en Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
import json, shutil, subprocess, sys
DRIVE_ROOT = Path('/content/drive/MyDrive/PBO Tesis')
REPO = Path('/content/backbone-pbo')
DATA_ROOT = DRIVE_ROOT / 'data/backpas'
SOURCE = DATA_ROOT / 'MIS'
if not (SOURCE / 'instance').is_dir() or not (SOURCE / 'backbone').is_dir():
    raise FileNotFoundError(f'Faltan datos MIS en {SOURCE}')
REPO.mkdir(parents=True, exist_ok=True)
for name in ('src', 'configs', 'scripts', 'pyproject.toml', 'uv.lock', 'README.md'):
    src = DRIVE_ROOT / name
    if src.is_dir():
        shutil.copytree(src, REPO / name, dirs_exist_ok=True)
    elif src.is_file():
        shutil.copy2(src, REPO / name)
(REPO / 'data').mkdir(exist_ok=True)
local_data = REPO / 'data/backpas'
if local_data.is_symlink():
    local_data.unlink()
elif local_data.exists() and local_data.resolve() != DATA_ROOT.resolve():
    shutil.rmtree(local_data)
if not local_data.exists():
    local_data.symlink_to(DATA_ROOT, target_is_directory=True)
BACKPAS = REPO / 'external/backpas'
if not (BACKPAS / 'src/3_train.py').is_file():
    BACKPAS.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/bryan-alvarado-ulloa/backpas.git', str(BACKPAS)], check=True)
    subprocess.run(['git', '-C', str(BACKPAS), 'checkout', '--detach', 'b5c633579fa73d673f3c4e65e177c89bcddea3d7'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[ml,backpas,data]'], cwd=REPO, check=True)
# El etiquetado explícito se ejecuta mediante GuroBack, no mediante un
# modelo alternativo escrito con la API Python del solver.
instance_count = sum(p.is_file() for p in (SOURCE / 'instance').iterdir())
backbone_count = sum(p.is_file() for p in (SOURCE / 'backbone').iterdir())
if instance_count != 1500 or backbone_count != 900:
    raise RuntimeError(f'MIS inesperado: {instance_count} instancias, {backbone_count} backbones')
print(f'MIS en Drive: {instance_count} instancias, {backbone_count} etiquetas.')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, torch
REPO = Path('/content/backbone-pbo')
DRIVE_ROOT = Path('/content/drive/MyDrive/PBO Tesis')
MATRIX = REPO / 'configs/pending_augmentation_jobs.json'
subprocess.run([sys.executable, 'scripts/materialize_pending_augmentations.py', '--repo', str(REPO)], cwd=REPO, check=True)
matrix = json.loads(MATRIX.read_text(encoding='utf-8'))
matrix['defaults'].update({'epochs': 65, 'threads': 4, 'batch_size': 32, 'batch_accumulation_size': 32, 'cuda': True})
MATRIX.write_text(json.dumps(matrix, indent=2) + '\n', encoding='utf-8')
shutil.copy2(MATRIX, DRIVE_ROOT / 'configs/pending_augmentation_jobs.json')
JOB = 'MIS/constraint-perturbation/seed-0'
row = next(j for j in matrix['jobs'] if j['problem'] + '/' + j['technique'] + '/seed-' + str(j['seed']) == JOB)
DATASET = REPO / row['dataset']
if DATASET.exists() and not (DATASET / 'augmentation.json').is_file():
    raise FileExistsError(f'Dataset previo sin manifiesto: {DATASET}')
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO / 'src') + ':' + env.get('PYTHONPATH', '')
cmd = [sys.executable, 'scripts/cloud_run.py', '--repo', str(REPO), '--matrix', str(MATRIX), '--job', JOB, '--drive-root', str(DRIVE_ROOT), '--resume']
if not torch.cuda.is_available():
    cmd.append('--cpu')
print('Dataset persistente:', DRIVE_ROOT / row['dataset'], flush=True)
subprocess.run(cmd, cwd=REPO, env=env, check=True)